In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Exploratory Data Analysis

In [2]:
data = pd.read_csv('ctd-datawiz-2023/train.csv')
X = data.drop(columns=['Attrition'])
y = data['Attrition']


In [3]:
X.describe()

,User ID,Mobile No.,Duration,Voicemail_Messages,Day_Minutes,Day_Calls,Day_Charge,Eve_Minutes,Eve_Calls,Eve_Charge,Night_Minutes,Night_Calls,Night_Charge,Intl_Minutes,Intl_Calls,Intl_Charge,Service_Calls
count,4150.000000,4.150000e+03,4150.000000,3681.000000,4150.000000,3640.000000,4150.000000,3626.000000,4150.000000,4150.000000,3548.000000,4150.000000,4150.000000,4150.000000,4150.000000,3561.000000,4150.000000
mean,2776.621928,7.988843e+09,99.967470,7.890519,177.145687,100.010440,391.499386,199.429923,100.250120,220.259219,199.746167,99.983855,116.795822,10.206482,4.455181,35.851342,1.495904
std,1346.549853,1.157382e+09,39.816089,13.599601,51.396781,19.569463,113.586282,50.015577,19.845768,55.418443,50.492036,19.898864,29.707936,2.759975,2.436228,9.615616,1.211282
min,406.000000,6.000109e+09,1.000000,0.000000,0.000000,0.000000,0.000000,22.300000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1589.250000,6.985293e+09,73.000000,0.000000,142.825000,87.000000,315.672500,165.125000,87.000000,182.292500,165.975000,87.000000,97.110000,8.500000,3.000000,29.900000,1.000000
50%,2830.500000,7.958110e+09,100.000000,0.000000,178.400000,100.000000,394.290000,199.800000,101.000000,220.805000,199.150000,100.000000,116.740000,10.300000,4.000000,36.140000,1.000000
75%,3962.750000,9.006551e+09,127.000000,18.000000,212.100000,113.000000,468.780000,232.900000,114.000000,257.400000,233.325000,113.000000,136.467500,12.000000,6.000000,42.120000,2.000000
max,5000.000000,9.998408e+09,243.000000,52.000000,351.500000,163.000000,776.880000,361.800000,170.000000,399.750000,395.000000,175.000000,231.010000,19.700000,19.000000,69.160000,8.000000


In [4]:
X.isna().sum()

User ID                 0
Mobile No.              0
User DOB                0
User Job                0
Duration                0
Intl_Plan             471
Voicemail_Plan          0
Voicemail_Messages    469
Day_Minutes             0
Day_Calls             510
Day_Charge              0
Eve_Minutes           524
Eve_Calls               0
Eve_Charge              0
Night_Minutes         602
Night_Calls             0
Night_Charge            0
Intl_Minutes            0
Intl_Calls              0
Intl_Charge           589
Service_Calls           0
dtype: int64

Areas of Note:
- 469 people got N/A as voicemail messages. This needs to be cleaned to most likely say 0 because these people did not have a voicemail plan. These 2 variables are likely to be highly correlated.
- 471 people have N/A on if they have an international plan (yes/no)
- 510 people have N/A as daycalls
- 524 people have N/A as eve_minutes
- 602 people have N/A as night_minutes
- 589 peopl have N/A for international charges


Somethings to clean data 

- if N/A for international charges and have no international plan -> 0, otherwise 1

- if N/A international plan and have international charges then plan ->yes otherwise no

- for the day calls find median time for day_minutes and divide the minites spent by that to find number of calls
    for eve_minutes and night_minutes do the opposite way

- if has voicemail plan -> median number of voicemessages, otherwise 0.




## Data Imputation

In [ ]:
from imputers import MasterImputer

imputer = MasterImputer(voicemail_strategy='zero', minute_strategy='mean', charge_strategy='mean'))

X = imputer.fit_transform(X)

In [6]:
X_test.isna().sum()

User ID                 0
Mobile No.              0
User DOB                0
User Job                0
Duration                0
Intl_Plan               0
Voicemail_Plan          0
Voicemail_Messages      0
Day_Minutes             0
Day_Calls             510
Day_Charge              0
Eve_Minutes             0
Eve_Calls               0
Eve_Charge              0
Night_Minutes           0
Night_Calls             0
Night_Charge            0
Intl_Minutes            0
Intl_Calls              0
Intl_Charge             0
Service_Calls           0
dtype: int64

In [7]:
cols = ['Night_Minutes', 'Night_Calls', 'Night_Charge']
df = X[cols][X['Night_Minutes'].isna()== False]


In [8]:
night_rate = np.mean(df['Night_Charge'] / df['Night_Minutes'])

In [9]:
X[X['Intl_Plan']== 'no'][5:10]

,User ID,Mobile No.,User DOB,User Job,Duration,Intl_Plan,Voicemail_Plan,Voicemail_Messages,Day_Minutes,Day_Calls,...,Eve_Minutes,Eve_Calls,Eve_Charge,Night_Minutes,Night_Calls,Night_Charge,Intl_Minutes,Intl_Calls,Intl_Charge,Service_Calls
9,3552,6442713170,1998-05-08,Catering manager,59,no,no,0.0,119.0,119.0,...,NaN,115,148.59,298.6,75,174.72,9.0,6,NaN,2
10,3555,8027097814,2004-10-24,Dramatherapist,72,no,no,0.0,129.8,106.0,...,188.7,138,208.52,212.5,116,124.28,8.3,4,NaN,4
12,3586,7695612579,1947-03-07,"Education officer, community",120,no,no,0.0,112.6,80.0,...,124.4,86,137.41,234.9,88,137.41,8.1,3,28.47,6
13,3588,8615106310,1980-10-14,Barrister's clerk,78,no,no,0.0,226.3,88.0,...,306.2,81,338.39,200.9,120,117.52,7.8,11,NaN,1
15,3602,9418050258,1951-05-11,Clinical research associate,166,no,no,0.0,156.8,100.0,...,171.5,93,189.54,149.1,97,87.23,11.2,10,39.26,1


In [10]:
X[X['Intl_Plan']== 'yes'][5:10]

,User ID,Mobile No.,User DOB,User Job,Duration,Intl_Plan,Voicemail_Plan,Voicemail_Messages,Day_Minutes,Day_Calls,...,Eve_Minutes,Eve_Calls,Eve_Charge,Night_Minutes,Night_Calls,Night_Charge,Intl_Minutes,Intl_Calls,Intl_Charge,Service_Calls
20,3617,6799631694,1965-11-12,Sport and exercise psychologist,127,yes,no,0.0,220.7,100.0,...,170.2,89,188.11,278.4,73,162.89,15.0,4,NaN,3
22,3629,7467984004,2001-01-15,Barrister,127,yes,no,0.0,324.3,107.0,...,141.6,90,156.52,NaN,90,121.42,12.8,7,44.98,1
23,3631,7639240780,1999-11-03,Chief Operating Officer,49,yes,yes,32.0,176.9,118.0,...,227.0,121,250.90,204.4,115,119.60,15.1,3,53.04,3
40,3770,7859581711,2002-03-26,Sub,162,yes,no,0.0,147.6,126.0,...,NaN,84,313.95,NaN,87,99.58,8.1,7,28.47,0
52,3910,9420849199,1961-11-27,"Civil engineer, contracting",155,yes,yes,NaN,234.6,84.0,...,214.9,119,237.51,196.7,69,115.05,8.0,2,28.08,1
